# Multimodal Image Retrieval

A local image vector store where a **text query returns the file path of the matching image** — the retrieval half of multimodal RAG. Covers the production strategies for turning images into searchable vectors, from simplest to most capable:

| # | Strategy | How it embeds an image | Library |
|---|----------|------------------------|---------|
| A | CLIP joint embeddings | image and text share one embedding space directly | `transformers` (CLIP) |
| B | Captioning + text embedding | a vision-language model writes a caption, then a text embedder embeds *that* | `transformers` (BLIP) + `sentence-transformers` |
| C | OCR + text embedding | extract literal text from the image, embed the text | `pytesseract` |
| D | Hosted multimodal embeddings | a managed API returns joint image/text vectors | `google-cloud-aiplatform` (Vertex AI) |
| E | Hybrid fusion | combine multiple signals above with reciprocal rank fusion | — |
| F | Late-interaction visual retrieval (ColPali-style) | conceptual — embed image patches directly, skip OCR entirely | (not run here — see notes) |

Each strategy indexes the same 12 synthetic sample images (`experiment/images/`) into its own local ChromaDB collection, so the final section can compare what each one actually retrieves for the same queries.


In [ ]:
%pip install -q torch transformers sentence-transformers
%pip install -q chromadb pillow matplotlib numpy pandas
# also needs the Tesseract OCR binary installed separately — see section C
%pip install -q pytesseract
# optional — only for section D
%pip install -q google-cloud-aiplatform


## Sample Image Library

12 synthetic images (`experiment/images/`) covering a few distinct categories on purpose, so different strategies can be shown succeeding/failing on different categories:

- **Shapes/colors** — `red_circle`, `blue_square`, `green_triangle`, `yellow_star`, `purple_circle` (CLIP's strength — it was trained on exactly this kind of visual-concept matching)
- **Scenes** — `sunset_gradient`, `ocean_waves` (also CLIP's strength — broad visual concepts)
- **A chart** — `bar_chart` (has both visual structure *and* readable text — good for comparing strategies)
- **Cartoon faces** — `cartoon_cat`, `cartoon_dog` (simple enough that captioning models may or may not get the animal right — realistic production behavior)
- **Text-heavy images** — `invoice_receipt`, `meeting_notes` (CLIP/captioning are weak here — this is where OCR earns its place)


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

IMAGE_DIR = Path("images")
image_paths = sorted(IMAGE_DIR.glob("*.png"))
print(f"{len(image_paths)} sample images: {[p.name for p in image_paths]}")

def show_images(paths: list, titles: list | None = None) -> None:
    paths = [Path(p) for p in paths]
    fig, axes = plt.subplots(1, len(paths), figsize=(3.2 * len(paths), 3.2))
    axes = [axes] if len(paths) == 1 else axes
    for ax, p, t in zip(axes, paths, titles or paths):
        ax.imshow(Image.open(p))
        ax.set_title(str(t), fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

show_images(image_paths[:4])


## Local Vector Store

One ChromaDB collection per strategy, same pattern as the earlier notebooks in this folder — lets every strategy be queried and compared independently.


In [ ]:
import chromadb

chroma_client = chromadb.PersistentClient(path="./chroma_db_images")

def index_collection(name: str, ids: list[str], embeddings, metadatas: list[dict], documents: list[str] | None = None):
    try:
        chroma_client.delete_collection(name)
    except Exception:
        pass
    collection = chroma_client.create_collection(name=name, metadata={"hnsw:space": "cosine"})
    collection.add(
        ids=ids,
        embeddings=[v.tolist() if hasattr(v, "tolist") else list(v) for v in embeddings],
        metadatas=metadatas,
        documents=documents or ids,
    )
    return collection

image_ids = [p.name for p in image_paths]


## A. CLIP Joint Embeddings

**What it is:** CLIP (Contrastive Language-Image Pretraining) is trained on hundreds of millions of (image, caption) pairs with a contrastive objective — pull matching image/text pairs together, push mismatched pairs apart — so images and text end up in the **same** embedding space. There's no intermediate text step: you embed the image directly, embed the query directly, and compare them with cosine similarity, exactly like the text-only retrieval in the earlier notebooks.

**When to use it:** the default choice for general visual-concept search ("a photo of a dog," "a red car," "people at a beach") — this is what almost every hosted "multimodal embeddings" API (§D) is doing under the hood, often with a proprietary CLIP variant. Weak on images that are mostly *text* (screenshots, scanned documents) since CLIP wasn't trained to read.

```
pip install transformers torch
```

⚠️ Downloads the CLIP model weights (~600MB) on first run.


In [ ]:
import numpy as np
import torch
from transformers import CLIPModel, CLIPProcessor

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()

def embed_images_clip(paths: list[Path]) -> np.ndarray:
    images = [Image.open(p).convert("RGB") for p in paths]
    inputs = clip_processor(images=images, return_tensors="pt")
    with torch.no_grad():
        features = clip_model.get_image_features(**inputs)
    return (features / features.norm(dim=-1, keepdim=True)).numpy()

def embed_text_clip(texts: list[str]) -> np.ndarray:
    inputs = clip_processor(text=texts, return_tensors="pt", padding=True)
    with torch.no_grad():
        features = clip_model.get_text_features(**inputs)
    return (features / features.norm(dim=-1, keepdim=True)).numpy()

clip_image_vecs = embed_images_clip(image_paths)
clip_collection = index_collection("images_clip", image_ids, clip_image_vecs, [{"path": str(p)} for p in image_paths])
print(f"indexed {len(image_ids)} images into 'images_clip', dim={clip_image_vecs.shape[1]}")


In [ ]:
def search_images_clip(query: str, k: int = 3) -> list[str]:
    query_vec = embed_text_clip([query])
    result = clip_collection.query(query_embeddings=query_vec.tolist(), n_results=k)
    return [m["path"] for m in result["metadatas"][0]]

for query in ["a red circle", "a sunset over water", "a bar chart"]:
    hits = search_images_clip(query, k=2)
    print(f"{query!r} -> {[Path(h).name for h in hits]}")


## B. Captioning + Text Embedding

**What it is:** run each image through an image-to-text (captioning) model to get a natural-language description, then embed *that description* with an ordinary text embedding model (the same `sentence-transformers` used throughout this folder). Retrieval becomes a text-to-text search — the image itself is never embedded directly.

**When to use it:** when you want the retriever to reason about image *content* in ways closer to how an LLM will describe/cite it in a generated answer (the caption doubles as the context you show the LLM at generation time), or when using a strong vision-language model (GPT-4V/Gemini-class, not just a small open captioner) for genuinely detailed descriptions of complex scenes CLIP's single embedding vector would blur together.

```
pip install transformers torch sentence-transformers
```

⚠️ Downloads the BLIP captioning model (~990MB) on first run.


In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration

blip_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip_model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")
blip_model.eval()

def caption_image(path: Path) -> str:
    image = Image.open(path).convert("RGB")
    inputs = blip_processor(image, return_tensors="pt")
    with torch.no_grad():
        out = blip_model.generate(**inputs, max_new_tokens=30)
    return blip_processor.decode(out[0], skip_special_tokens=True)

captions = {p.name: caption_image(p) for p in image_paths}
for name, caption in captions.items():
    print(f"{name:22s} -> {caption}")


In [ ]:
from sentence_transformers import SentenceTransformer

caption_embedder = SentenceTransformer("all-MiniLM-L6-v2")
caption_texts = [captions[p.name] for p in image_paths]
caption_vecs = caption_embedder.encode(caption_texts, normalize_embeddings=True)

caption_collection = index_collection(
    "images_caption", image_ids, caption_vecs,
    [{"path": str(p), "caption": captions[p.name]} for p in image_paths],
    documents=caption_texts,
)

def search_images_caption(query: str, k: int = 3) -> list[str]:
    query_vec = caption_embedder.encode([query], normalize_embeddings=True)
    result = caption_collection.query(query_embeddings=query_vec.tolist(), n_results=k)
    return [m["path"] for m in result["metadatas"][0]]

for query in ["a red circle", "a sunset over water", "a bar chart"]:
    hits = search_images_caption(query, k=2)
    print(f"{query!r} -> {[Path(h).name for h in hits]}")


## C. OCR + Text Embedding

**What it is:** for images that are mostly *text* (screenshots, scanned receipts, slide exports, whiteboard photos), extract the literal text with OCR (Tesseract) and embed that instead of — or alongside — a caption. Neither CLIP nor a captioning model reliably reads the specific text inside an image (a caption model might say "a printed invoice document" without ever mentioning the actual total due); OCR is the only strategy here that captures exact wording.

**When to use it:** document-heavy image corpora — invoices, forms, screenshots, scanned pages — where the query is about the image's *textual content*, not its visual appearance.

```
pip install pytesseract
```

⚠️ **`pytesseract` is a wrapper, not the OCR engine itself** — it needs the Tesseract binary installed separately:
- Windows: https://github.com/UB-Mannheim/tesseract/wiki
- macOS: `brew install tesseract`
- Linux: `apt-get install tesseract-ocr`

This section detects whether the binary is available and skips cleanly if it isn't, rather than failing the whole notebook.


In [ ]:
import pytesseract

try:
    pytesseract.get_tesseract_version()
    tesseract_available = True
except Exception:
    tesseract_available = False
    print("Tesseract binary not found — install it separately (see markdown above). Skipping OCR section.")

ocr_collection = None

if tesseract_available:
    def extract_text(path: Path) -> str:
        return pytesseract.image_to_string(Image.open(path)).strip()

    ocr_texts = {p.name: extract_text(p) for p in image_paths}
    ocr_hits = {name: text for name, text in ocr_texts.items() if len(text) > 5}
    for name, text in ocr_hits.items():
        print(f"{name}: {text[:70]!r}...")

    if ocr_hits:
        ocr_ids = list(ocr_hits.keys())
        ocr_paths = [p for p in image_paths if p.name in ocr_hits]
        ocr_vecs = caption_embedder.encode(list(ocr_hits.values()), normalize_embeddings=True)
        ocr_collection = index_collection(
            "images_ocr", ocr_ids, ocr_vecs,
            [{"path": str(p)} for p in ocr_paths],
            documents=list(ocr_hits.values()),
        )

def search_images_ocr(query: str, k: int = 3) -> list[str]:
    if ocr_collection is None:
        return []
    query_vec = caption_embedder.encode([query], normalize_embeddings=True)
    result = ocr_collection.query(query_embeddings=query_vec.tolist(), n_results=min(k, ocr_collection.count()))
    return [m["path"] for m in result["metadatas"][0]]

if tesseract_available:
    hits = search_images_ocr("what is the total amount due on the invoice", k=1)
    print(f"\nOCR search -> {[Path(h).name for h in hits]}")


## D. Hosted Multimodal Embeddings (Vertex AI)

**What it is:** the managed equivalent of §A — a hosted API (Vertex AI's `multimodalembedding@001`, similarly Amazon Titan Multimodal Embeddings, Azure AI Vision, Cohere `embed-v3` multimodal, Voyage `multimodal-3`) returns image and text embeddings in one shared space, without self-hosting or fine-tuning a CLIP-family model. Same underlying idea as §A, but no GPU/model management, billed per call — the tradeoff is the same local-vs-hosted one from the embedding methods notebook.

**When to use it:** production systems that don't want to own model serving infrastructure, or need multi-region/enterprise SLAs — most companies shipping multimodal search use a hosted API rather than self-hosting CLIP directly.

```
pip install google-cloud-aiplatform
```
Requires a `PROJECT_ID` with the Vertex AI API enabled — this cell is skipped automatically if it isn't set below.


In [ ]:
PROJECT_ID = ""  # TODO: set to your GCP project id to run this section
LOCATION = "us-central1"

if PROJECT_ID:
    import vertexai
    from vertexai.vision_models import Image as VertexImage
    from vertexai.vision_models import MultiModalEmbeddingModel

    vertexai.init(project=PROJECT_ID, location=LOCATION)
    mm_model = MultiModalEmbeddingModel.from_pretrained("multimodalembedding@001")

    def embed_image_vertex(path: Path) -> list[float]:
        return mm_model.get_embeddings(image=VertexImage.load_from_file(str(path))).image_embedding

    def embed_text_vertex(text: str) -> list[float]:
        return mm_model.get_embeddings(contextual_text=text).text_embedding

    vertex_vecs = np.array([embed_image_vertex(p) for p in image_paths])
    vertex_collection = index_collection("images_vertex", image_ids, vertex_vecs, [{"path": str(p)} for p in image_paths])

    def search_images_vertex(query: str, k: int = 3) -> list[str]:
        query_vec = np.array(embed_text_vertex(query))
        result = vertex_collection.query(query_embeddings=[query_vec.tolist()], n_results=k)
        return [m["path"] for m in result["metadatas"][0]]

    print(search_images_vertex("a red circle", k=2))
else:
    print("PROJECT_ID not set — skipping. Same idea as CLIP (§A), just hosted instead of local.")


## E. Hybrid Fusion — the Production Answer to "Which Strategy Should I Use"

**What it is:** none of the strategies above is strictly better — CLIP wins on visual concepts, captioning wins on scene description quality, OCR wins on exact text content. Production multimodal RAG systems typically run **multiple signals in parallel and fuse the rankings** (the same Reciprocal Rank Fusion used for hybrid dense+BM25 text search in `vertex_rag_pipeline.ipynb`), rather than betting everything on one embedding space.

`search_images()` below is the actual deliverable of this notebook: given a text query, it returns the **file path** of the best-matching image, fusing whichever signals are available (CLIP always; captions always; OCR only for the images where it found real text).


In [ ]:
def reciprocal_rank_fusion(rankings: list[list[str]], k: int = 60) -> list[str]:
    scores: dict[str, float] = {}
    for ranking in rankings:
        for rank, item in enumerate(ranking):
            scores[item] = scores.get(item, 0.0) + 1 / (k + rank + 1)
    return sorted(scores, key=scores.get, reverse=True)

def search_images(query: str, k: int = 3) -> list[str]:
    rankings = [search_images_clip(query, k=6), search_images_caption(query, k=6)]
    ocr_ranking = search_images_ocr(query, k=6)
    if ocr_ranking:
        rankings.append(ocr_ranking)
    return reciprocal_rank_fusion(rankings)[:k]

demo_queries = [
    "a red circle",
    "a chart showing quarterly revenue",
    "what is the total amount due on the invoice",
    "a cartoon animal face",
]
for query in demo_queries:
    results = search_images(query, k=1)
    print(f"query: {query!r}")
    print(f"  -> location: {results[0]}")


In [ ]:
# Visual sanity check — show the actual matched images for each query above.
for query in demo_queries:
    results = search_images(query, k=1)
    show_images(results, titles=[query])


## F. Late-Interaction Visual Document Retrieval (ColPali-style) — Conceptual

**What it is:** all five strategies above eventually reduce an image to text (a caption or OCR output) or to a single embedding vector (CLIP). ColPali (and the broader "vision-as-a-document" approach used by tools like Qdrant's and Vespa's multi-vector search) takes a different path for **document images specifically** (PDF pages, slides, scanned forms): the page image is split into patches, each patch gets its own embedding (via a vision-language model like PaliGemma), and retrieval uses **late interaction** — a MaxSim scoring function (from ColBERT) that compares every query-token embedding against every patch embedding — instead of collapsing everything to one vector per page.

This matters because OCR + text embedding **throws away layout** (tables, columns, where a number sits relative to its label), and a single CLIP/caption vector per page **throws away fine detail** (a page with 40 lines of text compresses to the same-sized vector as a single sentence). Late-interaction keeps both.

**When it's used in production:** PDF-heavy corpora with lots of tables, charts, and mixed layouts (financial reports, scientific papers, technical manuals) where OCR quality or layout loss has been measured to hurt retrieval — notably adopted by Vespa and Qdrant as a first-class multi-vector search pattern. Not implemented here — it needs a dedicated vision-language checkpoint (PaliGemma-based) and multi-vector storage most local ChromaDB setups aren't built for; treat this as the strategy to reach for once §A-E have been tried and layout loss is the measured bottleneck, not the default starting point.


## Summary

| Strategy | Reads exact text in image? | Understands visual concepts? | Needs | Production examples |
|---|---|---|---|---|
| A. CLIP joint embeddings | no | yes | GPU-friendly, ~600MB model | self-hosted CLIP/OpenCLIP deployments |
| B. Captioning + text embed | partially (short strings only) | yes, described in words | two models (captioner + text embedder) | LlamaIndex/LangChain multimodal pipelines using a VLM captioner |
| C. OCR + text embed | yes | no | Tesseract binary + text embedder | document/screenshot search, invoice/receipt pipelines |
| D. Hosted multimodal API | no | yes | API key/billing, no local GPU | Vertex AI, Amazon Titan, Azure AI Vision, Cohere, Voyage |
| E. Hybrid fusion | yes (via OCR signal) | yes (via CLIP/caption signal) | all of the above, fused | most real production multimodal search systems |
| F. Late-interaction (ColPali-style) | yes, with layout preserved | yes, at patch granularity | vision-language checkpoint + multi-vector store | Qdrant, Vespa document-image search |

**Rule of thumb:** start with CLIP (§A) or a hosted equivalent (§D) for general visual search. Add OCR (§C) the moment any image in the corpus is text-bearing (screenshots, scans, documents) — CLIP and captioning models both silently under-perform there, and it won't be obvious until specific queries mysteriously return nothing useful. Fuse signals (§E) rather than picking one, the same way hybrid dense+BM25 outperformed either alone for text retrieval. Reach for late-interaction/ColPali-style retrieval (§F) only once measurement shows layout loss is actually the bottleneck — it's meaningfully more infrastructure than everything else in this notebook.
